In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
import os

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [4]:
folder_path = 'raw_data'
file_pattern = os.path.join(folder_path, "BTS_*.csv")
all_files = glob.glob(file_pattern)

print(f"Found {len(all_files)} files. Joining...")
df_list = [pd.read_csv(file) for file in all_files]
combined_df = pd.concat(df_list, axis=0, ignore_index=True)

Found 11 files. Joining...


In [5]:
combined_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 6419315 entries, 0 to 6419314
Data columns (total 37 columns):
 #   Column               Dtype  
---  ------               -----  
 0   FL_DATE              str    
 1   OP_UNIQUE_CARRIER    str    
 2   OP_CARRIER           str    
 3   TAIL_NUM             str    
 4   OP_CARRIER_FL_NUM    int64  
 5   ORIGIN_AIRPORT_ID    int64  
 6   ORIGIN               str    
 7   ORIGIN_CITY_NAME     str    
 8   ORIGIN_STATE_ABR     str    
 9   ORIGIN_STATE_FIPS    int64  
 10  DEST_AIRPORT_ID      int64  
 11  DEST                 str    
 12  DEST_CITY_NAME       str    
 13  DEST_STATE_ABR       str    
 14  DEST_STATE_FIPS      int64  
 15  CRS_DEP_TIME         int64  
 16  DEP_TIME             float64
 17  DEP_DELAY            float64
 18  TAXI_OUT             float64
 19  WHEELS_OFF           float64
 20  TAXI_IN              float64
 21  CRS_ARR_TIME         int64  
 22  ARR_TIME             float64
 23  ARR_DELAY            float64
 24  CANCELLED

In [6]:
cols_to_drop = ['ORIGIN_STATE_FIPS', 'DEST_STATE_FIPS']
combined_df.drop(columns=[c for c in cols_to_drop if c in combined_df.columns], inplace=True)

In [7]:
combined_df.columns = combined_df.columns.str.strip()
df_obj_cols = combined_df.select_dtypes(['object']).columns
for col in df_obj_cols:
    combined_df[col] = combined_df[col].str.strip()

/var/folders/r5/7sdqfmvs4p987n2g0n_nng840000gn/T/ipykernel_63952/3530825980.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df_obj_cols = combined_df.select_dtypes(['object']).columns


In [8]:
if 'FL_DATE' in combined_df.columns:
    combined_df['FL_DATE'] = pd.to_datetime(
        combined_df['FL_DATE'],
        format='%m/%d/%Y %I:%M:%S %p',
        errors='coerce'
    )

In [9]:
if 'CANCELLATION_CODE' in combined_df.columns:
    combined_df['CANCELLATION_CODE'] = combined_df['CANCELLATION_CODE'].fillna('None')

In [10]:
int_cols = ['CANCELLED', 'DIVERTED', 'FLIGHTS', 'ORIGIN_AIRPORT_ID', 'DEST_AIRPORT_ID']
for col in int_cols:
    if col in combined_df.columns:
        combined_df[col] = combined_df[col].astype('Int64')

In [11]:
def format_military_time(time_val):
    if pd.isna(time_val):
        return np.nan
    return f"{int(time_val):04d}"

time_cols = ['CRS_DEP_TIME', 'DEP_TIME', 'WHEELS_OFF', 'TAXI_OUT', 'TAXI_IN', 'CRS_ARR_TIME', 'ARR_TIME']
for col in time_cols:
    if col in combined_df.columns:
        combined_df[col] = combined_df[col].apply(format_military_time)

In [12]:
print("Applying Lookups...")

# Load lookup tables
carriers_df = pd.read_csv('raw_data/L_UNIQUE_CARRIERS.csv')
airports_df = pd.read_csv('raw_data/L_AIRPORT_ID.csv')

# Create mapping dictionaries
# Note: Forcing airport codes to integers to ensure they match your integer columns perfectly
carrier_dict = dict(zip(carriers_df['Code'].str.strip(), carriers_df['Description'].str.strip()))
airport_dict = dict(zip(airports_df['Code'].astype(int), airports_df['Description'].str.strip()))

# Apply mappings to create new columns
combined_df['airline_name'] = combined_df['OP_UNIQUE_CARRIER'].map(carrier_dict)
combined_df['origin_airport_name'] = combined_df['ORIGIN_AIRPORT_ID'].map(airport_dict)
combined_df['dest_airport_name'] = combined_df['DEST_AIRPORT_ID'].map(airport_dict)

Applying Lookups...


In [13]:
parent_map = {
    'AA': 'American Group', 'MQ': 'American Group', 'OH': 'American Group',
    'DL': 'Delta Group', 'OO': 'Regional/Multiple',
    'UA': 'United Group', 'YX': 'Regional/Multiple'
}
combined_df['parent_airline'] = combined_df['OP_CARRIER'].map(parent_map).fillna(combined_df['OP_CARRIER'])

In [14]:
combined_df['OP_CARRIER'].value_counts()

OP_CARRIER
WN    1273810
DL     941193
AA     893794
OO     771007
UA     729406
YX     315833
MQ     273885
OH     228230
AS     224486
B6     210853
NK     182715
F9     181341
G4     119383
HA      73379
Name: count, dtype: int64

In [15]:
cancel_map = {
    'A': 'Carrier',
    'B': 'Weather',
    'C': 'National Air System',
    'D': 'Security'
}
if 'CANCELLATION_CODE' in combined_df.columns:
    combined_df['CANCELLATION_CODE'] = combined_df['CANCELLATION_CODE'].map(cancel_map)

In [16]:
lookup_cols_to_drop = ['ORIGIN_AIRPORT_ID', 'DEST_AIRPORT_ID', 'OP_CARRIER']
combined_df.drop(columns=[c for c in lookup_cols_to_drop if c in combined_df.columns], inplace=True)

In [17]:
rename_dict = {
    'FL_DATE': 'flight_date',
    'OP_UNIQUE_CARRIER': 'carrier_code',
    'TAIL_NUM': 'tail_number',
    'OP_CARRIER_FL_NUM': 'flight_number',
    'ORIGIN': 'origin_code',
    'ORIGIN_CITY_NAME': 'origin_city',
    'ORIGIN_STATE_ABR': 'origin_state',
    'DEST': 'dest_code',
    'DEST_CITY_NAME': 'dest_city',
    'DEST_STATE_ABR': 'dest_state',
    'CRS_DEP_TIME': 'scheduled_dep_time',
    'DEP_TIME': 'actual_dep_time',
    'DEP_DELAY': 'dep_delay_mins',
    'TAXI_OUT': 'taxi_out_mins',
    'WHEELS_OFF': 'wheels_off_time',
    'TAXI_IN': 'taxi_in_mins',
    'CRS_ARR_TIME': 'scheduled_arr_time',
    'ARR_TIME': 'actual_arr_time',
    'ARR_DELAY': 'arr_delay_mins',
    'CANCELLED': 'is_cancelled',
    'CANCELLATION_CODE': 'cancellation_reason',
    'DIVERTED': 'is_diverted',
    'CRS_ELAPSED_TIME': 'scheduled_flight_time_mins',
    'ACTUAL_ELAPSED_TIME': 'actual_flight_time_mins',
    'AIR_TIME': 'air_time_mins',
    'FLIGHTS': 'flight_count',
    'DISTANCE': 'distance_miles',
    'CARRIER_DELAY': 'carrier_delay_mins',
    'WEATHER_DELAY': 'weather_delay_mins',
    'NAS_DELAY': 'nas_delay_mins',
    'SECURITY_DELAY': 'security_delay_mins',
    'LATE_AIRCRAFT_DELAY': 'late_aircraft_delay_mins'
}
combined_df.rename(columns=rename_dict, inplace=True)

In [18]:
print("\n--- Data Cleaning Complete ---")
print(f"Total Rows: {len(combined_df)}")
print("\nSample Data:")
combined_df.head()


--- Data Cleaning Complete ---
Total Rows: 6419315

Sample Data:


,flight_date,carrier_code,tail_number,flight_number,origin_code,origin_city,origin_state,dest_code,dest_city,dest_state,scheduled_dep_time,actual_dep_time,dep_delay_mins,taxi_out_mins,wheels_off_time,taxi_in_mins,scheduled_arr_time,actual_arr_time,arr_delay_mins,is_cancelled,cancellation_reason,is_diverted,scheduled_flight_time_mins,actual_flight_time_mins,air_time_mins,flight_count,distance_miles,carrier_delay_mins,weather_delay_mins,nas_delay_mins,security_delay_mins,late_aircraft_delay_mins,airline_name,origin_airport_name,dest_airport_name,parent_airline
0,2025-07-01,AA,N101NN,307,LAX,"Los Angeles, CA",CA,JFK,"New York, NY",NY,0819,0812,-7.0,0022,0834,0012,1658,1712,14.0,0,NaN,0,339.0,360.0,326.0,1,2475.0,NaN,NaN,NaN,NaN,NaN,American Airlines Inc.,"Los Angeles, CA: Los Angeles International","New York, NY: John F. Kennedy International",American Group
1,2025-07-01,AA,N102NN,118,LAX,"Los Angeles, CA",CA,JFK,"New York, NY",NY,0600,NaN,NaN,NaN,NaN,NaN,1434,NaN,NaN,1,Weather,0,334.0,NaN,NaN,1,2475.0,NaN,NaN,NaN,NaN,NaN,American Airlines Inc.,"Los Angeles, CA: Los Angeles International","New York, NY: John F. Kennedy International",American Group
2,2025-07-01,AA,N102NN,15,JFK,"New York, NY",NY,SFO,"San Francisco, CA",CA,1700,NaN,NaN,NaN,NaN,NaN,2030,NaN,NaN,1,Weather,0,390.0,NaN,NaN,1,2586.0,NaN,NaN,NaN,NaN,NaN,American Airlines Inc.,"New York, NY: John F. Kennedy International","San Francisco, CA: San Francisco International",American Group
3,2025-07-01,AA,N102NN,276,SFO,"San Francisco, CA",CA,JFK,"New York, NY",NY,2245,NaN,NaN,NaN,NaN,NaN,0729,NaN,NaN,1,Weather,0,344.0,NaN,NaN,1,2586.0,NaN,NaN,NaN,NaN,NaN,American Airlines Inc.,"San Francisco, CA: San Francisco International","New York, NY: John F. Kennedy International",American Group
4,2025-07-01,AA,N102UW,1626,CLT,"Charlotte, NC",NC,OKC,"Oklahoma City, OK",OK,1125,1155,30.0,0018,1213,0005,1304,1316,12.0,0,NaN,0,159.0,141.0,118.0,1,940.0,NaN,NaN,NaN,NaN,NaN,American Airlines Inc.,"Charlotte, NC: Charlotte Douglas International","Oklahoma City, OK: Okc Will Rogers International",American Group


In [19]:
print("\nNormalizing data into relational tables...")

# 1. Airlines Table (Force unique carrier_code)
airlines_df = combined_df[['carrier_code', 'airline_name', 'parent_airline']].drop_duplicates(
    subset=['carrier_code'],
    keep='first'
).reset_index(drop=True)

origins = combined_df[['origin_code', 'origin_city', 'origin_state', 'origin_airport_name']].rename(columns={
    'origin_code': 'airport_code',
    'origin_city': 'city',
    'origin_state': 'state',
    'origin_airport_name': 'airport_name'
})

dests = combined_df[['dest_code', 'dest_city', 'dest_state', 'dest_airport_name']].rename(columns={
    'dest_code': 'airport_code',
    'dest_city': 'city',
    'dest_state': 'state',
    'dest_airport_name': 'airport_name'
})

airports_df = pd.concat([origins, dests]).drop_duplicates(
    subset=['airport_code'],
    keep='first'
).reset_index(drop=True)

cols_to_drop = [
    'airline_name', 'parent_airline',
    'origin_city', 'origin_state', 'origin_airport_name',
    'dest_city', 'dest_state', 'dest_airport_name', 'flight_count' # flight_count is always 1
]

flights_df = combined_df.drop(columns=cols_to_drop)

print("Normalization Complete. Primary Keys are strictly unique.")
print(f"Airlines Table: {len(airlines_df)} rows")
print(f"Airports Table: {len(airports_df)} rows")
print(f"Main Flights Table: {len(flights_df)} rows")


Normalizing data into relational tables...
Normalization Complete. Primary Keys are strictly unique.
Airlines Table: 14 rows
Airports Table: 351 rows
Main Flights Table: 6419315 rows


In [20]:
print("\nVerifying table relationships (Join Test)...")

test_join_df = flights_df.copy()

test_join_df = test_join_df.merge(
    airlines_df,
    on='carrier_code',
    how='left'
)

test_join_df = test_join_df.merge(
    airports_df.add_prefix('origin_'),
    left_on='origin_code',
    right_on='origin_airport_code',
    how='left'
).drop(columns=['origin_airport_code'])

test_join_df = test_join_df.merge(
    airports_df.add_prefix('dest_'),
    left_on='dest_code',
    right_on='dest_airport_code',
    how='left'
).drop(columns=['dest_airport_code'])

print(f"Original Row Count: {len(combined_df)}")
print(f"Rebuilt Row Count:  {len(test_join_df)}")

if len(combined_df) == len(test_join_df):
    print("✅ SUCCESS: Row counts match perfectly. No data duplication occurred.")
else:
    print("❌ ERROR: Row counts do not match.")

print("\nSample Rebuilt Row:")
print(test_join_df[['flight_date', 'airline_name', 'origin_city', 'dest_city', 'distance_miles']].head(1))


Verifying table relationships (Join Test)...
Original Row Count: 6419315
Rebuilt Row Count:  6419315
✅ SUCCESS: Row counts match perfectly. No data duplication occurred.

Sample Rebuilt Row:
  flight_date            airline_name      origin_city     dest_city  \
0  2025-07-01  American Airlines Inc.  Los Angeles, CA  New York, NY   

   distance_miles  
0          2475.0  


In [21]:
airports_df

,airport_code,city,state,airport_name
0,LAX,"Los Angeles, CA",CA,"Los Angeles, CA: Los Angeles International"
1,JFK,"New York, NY",NY,"New York, NY: John F. Kennedy International"
2,SFO,"San Francisco, CA",CA,"San Francisco, CA: San Francisco International"
3,CLT,"Charlotte, NC",NC,"Charlotte, NC: Charlotte Douglas International"
4,OKC,"Oklahoma City, OK",OK,"Oklahoma City, OK: Okc Will Rogers International"
...,...,...,...,...
346,ATY,"Watertown, SD",SD,"Watertown, SD: Watertown Regional"
347,PIR,"Pierre, SD",SD,"Pierre, SD: Pierre Regional"
348,LAF,"Lafayette, IN",IN,"Lafayette, IN: Purdue University"
349,ERI,"Erie, PA",PA,"Erie, PA: Erie International/Tom Ridge Field"


In [22]:
flights_df.to_csv("./clean_data/BTS_flights.csv", index=False, na_rep="")
airlines_df.to_csv("./clean_data/BTS_airlines.csv", index=False, na_rep="")
airports_df.to_csv("./clean_data/BTS_airports.csv", index=False, na_rep="")

In [26]:
flights_df['scheduled_flight_time_mins'].value_counts().sort_index()

scheduled_flight_time_mins
-99.0      1
-85.0      1
-67.0      1
-60.0      1
 9.0       3
          ..
 717.0     1
 739.0     1
 1348.0    1
 1358.0    1
 1510.0    1
Name: count, Length: 603, dtype: int64

In [27]:
flights_df[flights_df['scheduled_flight_time_mins'].isnull()]

,flight_date,carrier_code,tail_number,flight_number,origin_code,dest_code,scheduled_dep_time,actual_dep_time,dep_delay_mins,taxi_out_mins,wheels_off_time,taxi_in_mins,scheduled_arr_time,actual_arr_time,arr_delay_mins,is_cancelled,cancellation_reason,is_diverted,scheduled_flight_time_mins,actual_flight_time_mins,air_time_mins,distance_miles,carrier_delay_mins,weather_delay_mins,nas_delay_mins,security_delay_mins,late_aircraft_delay_mins
5819695,2025-06-26,HA,NaN,100,HNL,OGG,1305,NaN,NaN,NaN,NaN,NaN,1350,NaN,NaN,1,Carrier,0,NaN,NaN,NaN,100.0,NaN,NaN,NaN,NaN,NaN
5841130,2025-06-27,HA,NaN,89,BOS,HNL,0800,NaN,NaN,NaN,NaN,NaN,1305,NaN,NaN,1,Carrier,0,NaN,NaN,NaN,5095.0,NaN,NaN,NaN,NaN,NaN
